# AI Hedge Fund Simulator v5 — XGBoost + LightGBM Training

In [1]:
!pip install xgboost lightgbm pyarrow scikit-learn python-dateutil scipy --quiet
print('✅ Dependencies installed')

✅ Dependencies installed


## Cell 2 — Imports

In [2]:
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
import pickle
import os
import warnings
from datetime import datetime
from dateutil.relativedelta import relativedelta
from scipy.stats import spearmanr
from scipy.stats.mstats import winsorize
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

print(f'XGBoost  : {xgb.__version__}')
print(f'LightGBM : {lgb.__version__}')
print(f'pandas   : {pd.__version__}')
print(f'numpy    : {np.__version__}')

XGBoost  : 3.2.0
LightGBM : 4.6.0
pandas   : 2.2.2
numpy    : 2.0.2


## Cell 3 — Mount Google Drive & Paths

In [3]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE   = '/content/drive/MyDrive/Hedge_Fund_Simulator_V2'
RESULTS_DIR  = os.path.join(DRIVE_BASE, 'results', 'xgboost_v5')
MODELS_DIR   = os.path.join(DRIVE_BASE, 'models')
PARQUET_PATH = os.path.join(DRIVE_BASE, 'features_master_latest.parquet')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)

DATE_STAMP = datetime.today().strftime('%Y%m%d')
print(f'Date stamp  : {DATE_STAMP}')
print(f'Parquet     : {PARQUET_PATH}')
print(f'Results dir : {RESULTS_DIR}')
print(f'Models dir  : {MODELS_DIR}')

Mounted at /content/drive
Date stamp  : 20260502
Parquet     : /content/drive/MyDrive/Hedge_Fund_Simulator_V2/features_master_latest.parquet
Results dir : /content/drive/MyDrive/Hedge_Fund_Simulator_V2/results/xgboost_v5
Models dir  : /content/drive/MyDrive/Hedge_Fund_Simulator_V2/models


## Cell 4 — Configuration

In [4]:
TARGET_COL  = 'Target_Rank_21d'
TIER_COL    = 'Data_Tier'
DATE_COL    = 'Date'
TICKER_COL  = 'Ticker'

HOLDOUT_START = pd.Timestamp('2024-01-01')
TRAIN_YEARS   = 5
TEST_MONTHS   = 6
N_FOLDS       = 8
PURGE_DAYS    = 21

DECAY_HALF_LIFE_FINAL = 756
DECAY_HALF_LIFE_FOLD  = 504

WINSOR_LIMITS = (0.01, 0.01)

XGB_PARAMS = {
    'objective'        : 'reg:squarederror',
    'n_estimators'     : 1000,
    'max_depth'        : 6,
    'learning_rate'    : 0.05,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'min_child_weight' : 5,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 1.0,
    'random_state'     : 42,
    'n_jobs'           : -1,
    'tree_method'      : 'hist',
    'device'           : 'cuda',
}

LGB_PARAMS = {
    'objective'        : 'regression',
    'n_estimators'     : 1000,
    'max_depth'        : 6,
    'num_leaves'       : 63,
    'learning_rate'    : 0.05,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'min_child_samples': 20,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 1.0,
    'random_state'     : 42,
    'n_jobs'           : -1,
    'verbose'          : -1,
    'device'           : 'gpu',
    'importance_type'  : 'gain',
}

print('Config loaded.')
print(f'  CV: {TRAIN_YEARS}yr train / {TEST_MONTHS}mo test / {N_FOLDS} folds / {PURGE_DAYS}d purge gap')
print(f'  Hard holdout start         : {HOLDOUT_START.date()}')
print(f'  Decay half-life (full pool): {DECAY_HALF_LIFE_FINAL}d')
print(f'  Decay half-life (per fold) : {DECAY_HALF_LIFE_FOLD}d')
print(f'  Winsorization limits       : {WINSOR_LIMITS}')

Config loaded.
  CV: 5yr train / 6mo test / 8 folds / 21d purge gap
  Hard holdout start         : 2024-01-01
  Decay half-life (full pool): 756d
  Decay half-life (per fold) : 504d
  Winsorization limits       : (0.01, 0.01)


## Cell 5 — Load Data & Hard Holdout Split

In [5]:
print('Loading Parquet...')
df_all = pd.read_parquet(PARQUET_PATH)
df_all[DATE_COL] = pd.to_datetime(df_all[DATE_COL])

print(f'Raw shape          : {df_all.shape}')

df_all = df_all[df_all[TIER_COL].isin([1, 2])].copy()
print(f'After Tier A+B     : {len(df_all):,} rows')

df_all = df_all[df_all[TARGET_COL].notna()].copy()
print(f'After target filter: {len(df_all):,} rows')

holdout_df = df_all[df_all[DATE_COL] >= HOLDOUT_START].copy()
df         = df_all[df_all[DATE_COL] <  HOLDOUT_START].copy()

print()
print(f'Training pool : {df[DATE_COL].min().date()} → {df[DATE_COL].max().date()}'
      f'  |  {len(df):,} rows  |  {df[TICKER_COL].nunique()} tickers')
print(f'Holdout       : {holdout_df[DATE_COL].min().date()} → {holdout_df[DATE_COL].max().date()}'
      f'  |  {len(holdout_df):,} rows  |  {holdout_df[TICKER_COL].nunique()} tickers')

assert df[DATE_COL].max() < HOLDOUT_START, 'Training pool leaks into holdout!'
print('\n✅ Holdout isolation confirmed.')

Loading Parquet...
Raw shape          : (1289820, 75)
After Tier A+B     : 1,238,389 rows
After target filter: 1,230,367 rows

Training pool : 2010-01-04 → 2023-12-29  |  1,023,895 rows  |  382 tickers
Holdout       : 2024-01-01 → 2026-03-23  |  206,472 rows  |  382 tickers

✅ Holdout isolation confirmed.


## Cell 6 — Regime_Int Coverage Audit

In [6]:
print('Regime_Int coverage audit:')
print(f'  {"Year":<6} {"Total rows":>12} {"NULL %":>8} {"Value counts":<40}')
print(f'  {"-"*70}')

for year in sorted(df_all[DATE_COL].dt.year.unique()):
    subset     = df_all[df_all[DATE_COL].dt.year == year]
    null_pct   = subset['Regime_Int'].isna().mean() * 100
    val_counts = subset['Regime_Int'].value_counts().to_dict()
    val_str    = str({int(k): v for k, v in sorted(val_counts.items())})
    flag = '  ⚠ HIGH NULL' if null_pct > 50 else ''
    print(f'  {year:<6} {len(subset):>12,} {null_pct:>7.1f}%  {val_str}{flag}')

print()
train_null = df['Regime_Int'].isna().mean() * 100
hold_null  = holdout_df['Regime_Int'].isna().mean() * 100
print(f'Training pool Regime_Int null: {train_null:.1f}%')
print(f'Holdout       Regime_Int null: {hold_null:.1f}%')
if hold_null > 20:
    print('⚠ WARNING: High Regime_Int null in holdout — HMM retrain recommended.')
else:
    print('✅ Regime_Int coverage acceptable.')

Regime_Int coverage audit:
  Year     Total rows   NULL % Value counts                            
  ----------------------------------------------------------------------
  2010         56,395     5.9%  {1: 11095, 2: 30664, 3: 11317}
  2011         59,285     1.2%  {1: 28307, 2: 29293, 3: 947}
  2012         61,954     2.0%  {0: 3289, 1: 5286, 2: 29307, 3: 22810}
  2013         61,825     0.4%  {0: 9107, 1: 9719, 2: 27944, 3: 14802}
  2014         62,070     0.8%  {0: 9717, 1: 6786, 2: 2614, 3: 42439}
  2015         65,796     1.2%  {1: 5130, 2: 23559, 3: 36304}
  2016         68,380     0.8%  {0: 8657, 1: 1932, 2: 19771, 3: 37468}
  2017         72,811     0.0%  {0: 66475, 3: 6336}
  2018         77,178     0.4%  {0: 36122, 2: 10925, 3: 29825}
  2019         80,663     1.2%  {0: 11075, 1: 11310, 2: 11575, 3: 45727}
  2020         83,623     0.0%  {0: 1002, 1: 27377, 2: 43817, 3: 11427}
  2021         87,331     0.0%  {0: 23379, 1: 7690, 2: 36444, 3: 19818}
  2022         93,294     0

## Cell 7 — Column Taxonomy: Flags, Features, Targets

In [13]:
META_COLS = {
    'id', 'Ticker', 'Date',
    'Target_Return_21d', 'Target_Rank_21d',
    'Target_Direction_Median', 'Target_Direction_Tertile', 'Target_Vol_5d',
    'Data_Tier',
}

FLAG_COLS = {
    'Price_Gap_Flag',
    'SMA200_Available',
    'SMA50_Available',
    'ADX14_Available',
    'Volatility20d_Available',
    'Fundamentals_Available',
    'Gross_Margin_Is_Proxy',
}

RAW_PRICE_COLS = {
    'Open', 'High', 'Low', 'Close', 'Adj_Close',
    'Volume', 'VWAP_Daily', 'ADV_20d_Cr', 'OBV',
    'SMA_20', 'SMA_50', 'SMA_200', 'EMA_9', 'EMA_21',
    'BB_Upper', 'BB_Middle', 'BB_Lower',
    'MACD', 'MACD_Signal',
    'Stoch_D',
    'EPS_Basic',
    'India_VIX', 'USDINR', 'Crude_Oil', 'Gold',
    'FII_Momentum_5d', 'DII_Momentum_5d',
    'RepoRate_x_DebtEquity',
}

EXCLUDE_COLS = META_COLS | FLAG_COLS | RAW_PRICE_COLS

FEATURE_COLS_RAW = [c for c in df.columns if c not in EXCLUDE_COLS]

for col in FEATURE_COLS_RAW:
    if df[col].dtype == object:
        df[col]         = pd.to_numeric(df[col], errors='coerce')
        holdout_df[col] = pd.to_numeric(holdout_df[col], errors='coerce')

FEATURE_COLS = [c for c in FEATURE_COLS_RAW
                if pd.api.types.is_numeric_dtype(df[c])]

dropped_non_numeric = set(FEATURE_COLS_RAW) - set(FEATURE_COLS)
if dropped_non_numeric:
    print(f'Dropped non-numeric cols: {dropped_non_numeric}')

assert TARGET_COL               not in FEATURE_COLS, 'TARGET LEAKAGE'
assert 'Target_Return_21d'      not in FEATURE_COLS, 'TARGET RETURN LEAKAGE'
assert 'Fundamentals_Available' not in FEATURE_COLS, 'FLAG IN FEATURES'
assert 'Gross_Margin_Is_Proxy'  not in FEATURE_COLS, 'FLAG IN FEATURES'
assert 'Adj_Close'              not in FEATURE_COLS, 'RAW PRICE IN FEATURES'
assert 'SMA_20'                 not in FEATURE_COLS, 'RAW SMA IN FEATURES'

NEW_V23_COLS = [
    'Mom_12_1', 'Mom_6_1',
    'Mom_1M_Rank', 'Mom_12_1_Rank', 'Mom_6_1_Rank', 'RS_21d_Rank',
    'ROA_Sector_Rank', 'EBITDA_Margin_Sector_Rank', 'Revenue_Growth_Sector_Rank',
]
present_new  = [c for c in NEW_V23_COLS if c in FEATURE_COLS]
missing_new  = [c for c in NEW_V23_COLS if c not in FEATURE_COLS]
print(f'\n new feature columns present in Parquet : {len(present_new)}/9')
if missing_new:
    print(f'  ⚠ MISSING (re-export features_master_latest.parquet after running features.py ):')
    for c in missing_new:
        print(f'    - {c}')
else:
    print('  ✅ All 9 new columns confirmed in feature set')

print(f'\n✅ Feature columns confirmed: {len(FEATURE_COLS)}')



FIX-A/B new feature columns present in Parquet : 9/9
  ✅ All 9 new columns confirmed in feature set

✅ Feature columns confirmed: 60


## Cell 8 — Flag-Based Null Imputation

In [8]:
FUNDAMENTAL_COLS = [
    'Revenue_YoY_Growth', 'EBITDA_Margin', 'Net_Margin', 'FCF_Margin',
    'Asset_Turnover', 'ROA', 'Debt_to_Equity',
    'Gross_Profit_Margin', 'Debt_to_Assets', 'OCF_to_Net_Income',
    'Delta_ROA', 'Delta_DebtEquity', 'Rel_ROA', 'Rel_EBITDA_Margin',
    'ROA_Sector_Rank', 'EBITDA_Margin_Sector_Rank', 'Revenue_Growth_Sector_Rank',
]
GAP_AFFECTED   = ['Return_5d', 'Return_21d', 'Return_60d', 'RS_21d', 'RS_60d']

MOMENTUM_RANK_COLS_252 = ['Mom_12_1', 'Mom_12_1_Rank', 'Mom_1M_Rank', 'RS_21d_Rank']
MOMENTUM_RANK_COLS_126 = ['Mom_6_1', 'Mom_6_1_Rank']

FLAG_IMPUTE_MAP = {
    'Fundamentals_Available' : (0, FUNDAMENTAL_COLS),
    'SMA200_Available'       : (0, ['Price_to_SMA200'] + MOMENTUM_RANK_COLS_252),
    'SMA50_Available'        : (0, ['Price_to_SMA50']  + MOMENTUM_RANK_COLS_126),
    'ADX14_Available'        : (0, ['ADX_14']),
    'Volatility20d_Available': (0, ['Volatility_20d']),
    'Price_Gap_Flag'         : (1, GAP_AFFECTED),
}

def compute_impute_medians(df_train: pd.DataFrame) -> dict:
    medians = {}
    for flag_col, (missing_val, cols) in FLAG_IMPUTE_MAP.items():
        available_val = 1 - missing_val
        if flag_col not in df_train.columns:
            continue
        available_mask = df_train[flag_col] == available_val
        for col in cols:
            if col not in df_train.columns:
                continue
            series = df_train.loc[available_mask, col].dropna()
            medians[(flag_col, col)] = series.median() if len(series) > 0 else 0.0
    return medians

def apply_imputation(X: pd.DataFrame,
                     source_df: pd.DataFrame,
                     medians: dict) -> pd.DataFrame:
    X = X.copy()
    for flag_col, (missing_val, cols) in FLAG_IMPUTE_MAP.items():
        if flag_col not in source_df.columns:
            continue
        missing_mask = source_df[flag_col] == missing_val
        for col in cols:
            if col not in X.columns:
                continue
            median_val = medians.get((flag_col, col), np.nan)
            fill_mask  = missing_mask & X[col].isna()
            X.loc[fill_mask, col] = median_val
    return X

IMPUTE_MEDIANS = compute_impute_medians(df)

print('Imputation medians computed from training pool:')
print(f'  {len(IMPUTE_MEDIANS)} (flag, column) pairs')
print()
print(f'  {"Flag":<28} {"Column":<35} {"Median":>10}')
print(f'  {"-"*28} {"-"*35} {"-"*10}')
for (flag_col, col), val in sorted(IMPUTE_MEDIANS.items()):
    print(f'  {flag_col:<28} {col:<35} {val:>10.4f}')

Imputation medians computed from training pool:
  32 (flag, column) pairs

  Flag                         Column                                  Median
  ---------------------------- ----------------------------------- ----------
  ADX14_Available              ADX_14                                 23.9692
  Fundamentals_Available       Asset_Turnover                          0.6921
  Fundamentals_Available       Debt_to_Assets                          0.1810
  Fundamentals_Available       Debt_to_Equity                          0.4267
  Fundamentals_Available       Delta_DebtEquity                       -0.0055
  Fundamentals_Available       Delta_ROA                               0.0003
  Fundamentals_Available       EBITDA_Margin                           0.1685
  Fundamentals_Available       EBITDA_Margin_Sector_Rank               0.5556
  Fundamentals_Available       FCF_Margin                              0.0606
  Fundamentals_Available       Gross_Profit_Margin                 

## Cell 9 — Z-Score Scaling Groups

In [9]:
NO_SCALE_COLS = {
    'RSI_14', 'Stoch_K',

    'Return_1d', 'Return_5d', 'Return_21d', 'Return_60d', 'Log_Return_1d',
    'RS_21d', 'RS_60d',

    'Mom_12_1', 'Mom_6_1',

    'Mom_1M_Rank', 'Mom_12_1_Rank', 'Mom_6_1_Rank', 'RS_21d_Rank',
    'ROA_Sector_Rank', 'EBITDA_Margin_Sector_Rank', 'Revenue_Growth_Sector_Rank',

    'Volatility_20d', 'Volume_Ratio_20d',

    'Price_to_SMA20', 'Price_to_SMA50', 'Price_to_SMA200',
    'Price_to_52W_High', 'Price_to_52W_Low',

    'BB_Width', 'BB_PctB', 'VWAP_Dev',

    'MACD_Hist',

    'OBV_Change_5d',

    'Revenue_YoY_Growth', 'EBITDA_Margin', 'Net_Margin', 'FCF_Margin',
    'Asset_Turnover', 'ROA', 'Gross_Profit_Margin', 'Debt_to_Assets',

    'Delta_ROA', 'Delta_DebtEquity', 'Rel_ROA', 'Rel_EBITDA_Margin',

    'USDINR_x_Revenue_Growth', 'Repo_Rate_Change',

    'Regime_Int',
}

SCALE_COLS = [c for c in FEATURE_COLS if c not in NO_SCALE_COLS]

print(f'Total feature columns : {len(FEATURE_COLS)}')
print(f'Columns to z-score    : {len(SCALE_COLS)}')
print(f'Columns left as-is    : {len(FEATURE_COLS) - len(SCALE_COLS)}')
print()
print('Columns that WILL be winsorized + z-scored:')
for c in sorted(SCALE_COLS):
    print(f'  {c}')

rank_in_scale = [c for c in SCALE_COLS if 'Rank' in c or c in ('Mom_12_1', 'Mom_6_1')]
if rank_in_scale:
    print(f'\n⚠ WARNING: rank/momentum cols in SCALE_COLS: {rank_in_scale}')
else:
    print('\n✅ No rank or momentum columns in SCALE_COLS — correct.')

Total feature columns : 60
Columns to z-score    : 15
Columns left as-is    : 45

Columns that WILL be winsorized + z-scored:
  ADX_14
  ATR_14
  Announcement_Score
  Beta_to_Crude
  Beta_to_DII
  Beta_to_FII
  Beta_to_Gold
  Beta_to_Nifty50
  Beta_to_Nifty500
  Beta_to_USDINR
  Beta_to_VIX
  Debt_to_Equity
  OCF_to_Net_Income
  Sentiment_Available
  Sentiment_Score_Lag1

✅ No rank or momentum columns in SCALE_COLS — correct.


## Cell 10 — Build Walk-Forward CV Folds

In [10]:
data_start = df[DATE_COL].min()
data_end   = df[DATE_COL].max()

print(f'Training pool: {data_start.date()} → {data_end.date()}')
print(f'Purge gap    : {PURGE_DAYS} trading days between train_end and test_start')
print()

folds = []
for i in range(N_FOLDS):
    test_end    = data_end   - relativedelta(months=TEST_MONTHS * i)
    test_start  = test_end   - relativedelta(months=TEST_MONTHS)
    train_end   = test_start - pd.Timedelta(days=PURGE_DAYS)
    train_start = train_end  - relativedelta(years=TRAIN_YEARS)

    if train_start < data_start:
        print(f'Fold {i+1}: train_start {train_start.date()} < data_start '
              f'{data_start.date()} — stopping at {len(folds)} folds')
        break

    folds.append({
        'fold'        : i + 1,
        'train_start' : train_start,
        'train_end'   : train_end,
        'test_start'  : test_start,
        'test_end'    : test_end,
    })

folds = folds[::-1]
for i, f in enumerate(folds):
    f['fold'] = i + 1

for f in folds:
    assert f['test_end'] <= HOLDOUT_START, f"Fold {f['fold']} test leaks into holdout!"
    assert f['train_end'] < f['test_start'], f"Fold {f['fold']} purge gap violated!"

print(f'Folds confirmed: {len(folds)}')
print()
print(f'  {"Fold":<6} {"Train Start":<13} {"Train End":<13} '
      f'{"Purge →":<10} {"Test Start":<13} {"Test End":<13}')
print(f'  {"-"*68}')
for f in folds:
    gap_days = (f['test_start'] - f['train_end']).days
    print(f'  {f["fold"]:<6} {str(f["train_start"].date()):<13} '
          f'{str(f["train_end"].date()):<13} '
          f'{gap_days:>4}d      '
          f'{str(f["test_start"].date()):<13} '
          f'{str(f["test_end"].date()):<13}')

Training pool: 2010-01-04 → 2023-12-29
Purge gap    : 21 trading days between train_end and test_start

Folds confirmed: 8

  Fold   Train Start   Train End     Purge →    Test Start    Test End     
  --------------------------------------------------------------------
  1      2014-12-08    2019-12-08      21d      2019-12-29    2020-06-29   
  2      2015-06-08    2020-06-08      21d      2020-06-29    2020-12-29   
  3      2015-12-08    2020-12-08      21d      2020-12-29    2021-06-29   
  4      2016-06-08    2021-06-08      21d      2021-06-29    2021-12-29   
  5      2016-12-08    2021-12-08      21d      2021-12-29    2022-06-29   
  6      2017-06-08    2022-06-08      21d      2022-06-29    2022-12-29   
  7      2017-12-08    2022-12-08      21d      2022-12-29    2023-06-29   
  8      2018-06-08    2023-06-08      21d      2023-06-29    2023-12-29   


## Cell 11 — Helper Functions

In [14]:
def compute_ic(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Global Spearman rank IC across all stocks × dates."""
    ic, _ = spearmanr(y_true, y_pred)
    return float(ic)

def compute_ic_per_date(df_eval: pd.DataFrame,
                        pred_col: str = 'pred') -> tuple:

    daily_ics = (
        df_eval.groupby(DATE_COL)
        .apply(lambda g: spearmanr(g[TARGET_COL], g[pred_col])[0]
               if len(g) >= 5 else np.nan)
        .dropna()
    )
    mean_ic = float(daily_ics.mean())
    ic_std  = float(daily_ics.std())
    icir    = mean_ic / ic_std if ic_std > 1e-10 else np.nan
    return mean_ic, icir, daily_ics

def compute_time_decay_weights(dates: pd.Series,
                               half_life_days: int) -> np.ndarray:

    days_from_end = (dates.max() - dates).dt.days.values
    weights       = np.exp(-np.log(2) * days_from_end / half_life_days)
    weights       = weights / weights.mean()
    return weights.astype(np.float32)

def winsorize_dataframe(X: pd.DataFrame, cols: list,
                        limits: tuple = (0.01, 0.01)) -> pd.DataFrame:

    X = X.copy()
    for col in cols:
        if col not in X.columns:
            continue
        nan_mask   = X[col].isna()
        col_median = X[col].median()
        filled     = X[col].fillna(col_median).to_numpy(dtype=np.float64)
        winsorized = winsorize(filled, limits=limits).data
        X[col]     = winsorized.astype(np.float64)
        X.loc[nan_mask, col] = np.nan
    return X

def prepare_fold_data(df: pd.DataFrame, fold: dict):

    train_mask = (
        (df[DATE_COL] >= fold['train_start']) &
        (df[DATE_COL] <  fold['train_end'])
    )
    test_mask = (
        (df[DATE_COL] >= fold['test_start']) &
        (df[DATE_COL] <  fold['test_end'])
    )
    train_df = df[train_mask].copy()
    test_df  = df[test_mask].copy()

    fold_medians = compute_impute_medians(train_df)

    X_train_raw = apply_imputation(train_df[FEATURE_COLS], train_df, fold_medians)
    X_test_raw  = apply_imputation(test_df[FEATURE_COLS],  test_df,  fold_medians)

    y_train = train_df[TARGET_COL].values
    y_test  = test_df[TARGET_COL].values

    active_scale  = [c for c in SCALE_COLS if c in X_train_raw.columns]
    X_train_winsr = winsorize_dataframe(X_train_raw, active_scale, WINSOR_LIMITS)
    X_test_winsr  = winsorize_dataframe(X_test_raw,  active_scale, WINSOR_LIMITS)

    scaler    = StandardScaler()
    X_train_s = X_train_winsr.copy()
    X_test_s  = X_test_winsr.copy()
    X_train_s[active_scale] = scaler.fit_transform(X_train_winsr[active_scale])
    X_test_s[active_scale]  = scaler.transform(X_test_winsr[active_scale])

    sample_weights = compute_time_decay_weights(
        train_df[DATE_COL], DECAY_HALF_LIFE_FOLD
    )

    return X_train_s, y_train, X_test_s, y_test, test_df, scaler, fold_medians, sample_weights

print('✅ Helper functions defined.')

✅ Helper functions defined.


## Cell 12 — Walk-Forward CV: XGBoost + LightGBM

In [15]:
xgb_cv_results = []
lgb_cv_results = []

for fold in folds:
    fn = fold['fold']
    print(f'\n── Fold {fn}/{len(folds)} '
          f'train {fold["train_start"].date()}→{fold["train_end"].date()} '
          f'test {fold["test_start"].date()}→{fold["test_end"].date()} ──')

    X_train, y_train, X_test, y_test, test_df, scaler, fold_medians, sw = \
        prepare_fold_data(df, fold)

    print(f'  Train: {len(X_train):,} rows | Test: {len(X_test):,} rows | '
          f'Features: {X_train.shape[1]}')
    print(f'  Weight range: [{sw.min():.3f}, {sw.max():.3f}]  mean={sw.mean():.3f}')

    train_null_pct = X_train.isna().mean().max() * 100
    test_null_pct  = X_test.isna().mean().max()  * 100
    print(f'  Max null% after imputation — train: {train_null_pct:.1f}%  '
          f'test: {test_null_pct:.1f}%')

    xgb_params = XGB_PARAMS.copy()
    try:
        xgb_model = xgb.XGBRegressor(**xgb_params)
        xgb_model.fit(X_train, y_train, sample_weight=sw, verbose=False)
    except Exception as e:
        print(f'  XGBoost CUDA failed ({e}), retrying on CPU...')
        xgb_params['device'] = 'cpu'
        xgb_model = xgb.XGBRegressor(**xgb_params)
        xgb_model.fit(X_train, y_train, sample_weight=sw, verbose=False)

    xgb_preds         = xgb_model.predict(X_test)
    xgb_ic_global     = compute_ic(y_test, xgb_preds)
    _xgb_eval         = test_df[[DATE_COL, TARGET_COL]].copy()
    _xgb_eval['pred'] = xgb_preds
    xgb_mean_ic, xgb_icir, _ = compute_ic_per_date(_xgb_eval)

    xgb_cv_results.append({
        'fold': fn, 'ic': xgb_ic_global,
        'mean_ic': xgb_mean_ic, 'icir': xgb_icir,
        'model': xgb_model, 'scaler': scaler, 'medians': fold_medians,
        'n_train': len(X_train), 'n_test': len(X_test),
    })
    print(f'  XGBoost  — Global IC: {xgb_ic_global:+.4f} | '
          f'Per-date IC: {xgb_mean_ic:+.4f} | ICIR: {xgb_icir:.3f}')

    lgb_params = LGB_PARAMS.copy()
    try:
        lgb_model = lgb.LGBMRegressor(**lgb_params)
        lgb_model.fit(X_train, y_train, sample_weight=sw)
    except Exception as e:
        print(f'  LightGBM GPU failed ({e}), retrying on CPU...')
        lgb_params['device'] = 'cpu'
        lgb_model = lgb.LGBMRegressor(**lgb_params)
        lgb_model.fit(X_train, y_train, sample_weight=sw)

    lgb_preds         = lgb_model.predict(X_test)
    lgb_ic_global     = compute_ic(y_test, lgb_preds)
    _lgb_eval         = test_df[[DATE_COL, TARGET_COL]].copy()
    _lgb_eval['pred'] = lgb_preds
    lgb_mean_ic, lgb_icir, _ = compute_ic_per_date(_lgb_eval)

    lgb_cv_results.append({
        'fold': fn, 'ic': lgb_ic_global,
        'mean_ic': lgb_mean_ic, 'icir': lgb_icir,
        'model': lgb_model,
        'n_train': len(X_train), 'n_test': len(X_test),
    })
    print(f'  LightGBM — Global IC: {lgb_ic_global:+.4f} | '
          f'Per-date IC: {lgb_mean_ic:+.4f} | ICIR: {lgb_icir:.3f}')

print('\n✅ Walk-forward CV complete.')


── Fold 1/8 train 2014-12-08→2019-12-08 test 2019-12-29→2020-06-29 ──
  Train: 363,953 rows | Test: 41,132 rows | Features: 60
  Weight range: [0.212, 2.602]  mean=1.000
  Max null% after imputation — train: 100.0%  test: 100.0%
  XGBoost  — Global IC: +0.1323 | Per-date IC: +0.1397 | ICIR: 0.939
  LightGBM — Global IC: +0.1214 | Per-date IC: +0.1268 | ICIR: 0.846

── Fold 2/8 train 2015-06-08→2020-06-08 test 2020-06-29→2020-12-29 ──
  Train: 372,160 rows | Test: 42,124 rows | Features: 60
  Weight range: [0.212, 2.610]  mean=1.000
  Max null% after imputation — train: 100.0%  test: 100.0%
  XGBoost  — Global IC: +0.0175 | Per-date IC: +0.0179 | ICIR: 0.272
  LightGBM — Global IC: +0.0172 | Per-date IC: +0.0187 | ICIR: 0.298

── Fold 3/8 train 2015-12-08→2020-12-08 test 2020-12-29→2021-06-29 ──
  Train: 381,417 rows | Test: 43,206 rows | Features: 60
  Weight range: [0.213, 2.619]  mean=1.000
  Max null% after imputation — train: 100.0%  test: 100.0%
  XGBoost  — Global IC: +0.0503 | 

## Cell 13 — CV Summary & IC Grade

In [16]:
xgb_ics      = [r['ic']      for r in xgb_cv_results]
lgb_ics      = [r['ic']      for r in lgb_cv_results]
xgb_mean_ics = [r['mean_ic'] for r in xgb_cv_results]
lgb_mean_ics = [r['mean_ic'] for r in lgb_cv_results]
xgb_icirs    = [r['icir']    for r in xgb_cv_results]
lgb_icirs    = [r['icir']    for r in lgb_cv_results]

W = 80
print('=' * W)
print('WALK-FORWARD CV RESULTS')
print('=' * W)
print(f'  {"Fold":<5} {"XGB Global":>11} {"LGB Global":>11} '
      f'{"XGB PerDate":>12} {"LGB PerDate":>12} {"XGB ICIR":>9} {"LGB ICIR":>9}')
print(f'  {"-"*71}')
for xr, lr in zip(xgb_cv_results, lgb_cv_results):
    fold_flag = ''
    if xr['fold'] == 1:  fold_flag = ' ← COVID (regime-break, excluded from honest baseline)'
    if xr['fold'] == 6:  fold_flag = ' ← 2022 rate-shock (regime-break, excluded from honest baseline)'
    if xr['fold'] == 8:  fold_flag = ' ← 2023 chop (regime-break, excluded from honest baseline)'
    print(f'  {xr["fold"]:<5} {xr["ic"]:>+11.4f} {lr["ic"]:>+11.4f} '
          f'{xr["mean_ic"]:>+12.4f} {lr["mean_ic"]:>+12.4f} '
          f'{xr["icir"]:>9.3f} {lr["icir"]:>9.3f}{fold_flag}')
print(f'  {"-"*71}')
print(f'  {"Mean":<5} {np.mean(xgb_ics):>+11.4f} {np.mean(lgb_ics):>+11.4f} '
      f'{np.nanmean(xgb_mean_ics):>+12.4f} {np.nanmean(lgb_mean_ics):>+12.4f} '
      f'{np.nanmean(xgb_icirs):>9.3f} {np.nanmean(lgb_icirs):>9.3f}')
print(f'  {"Std":<5} {np.std(xgb_ics):>11.4f} {np.std(lgb_ics):>11.4f}')
print(f'  {"Min":<5} {np.min(xgb_ics):>+11.4f} {np.min(lgb_ics):>+11.4f}')
print(f'  {"Max":<5} {np.max(xgb_ics):>+11.4f} {np.max(lgb_ics):>+11.4f}')
print('=' * W)

mean_ic_pd = np.nanmean(xgb_mean_ics)

REGIME_BREAK_FOLDS = {1, 6, 8}
normal_fold_ics = [
    xgb_mean_ics[i] for i, r in enumerate(xgb_cv_results)
    if r['fold'] not in REGIME_BREAK_FOLDS
]
mean_ic_ex_covid    = np.nanmean(xgb_mean_ics[1:])
mean_ic_ex_covid_f6 = np.nanmean(xgb_mean_ics[1:5] + xgb_mean_ics[6:])
mean_ic_honest      = np.nanmean(normal_fold_ics)

print(f'\nXGBoost per-date mean IC (all folds)                : {mean_ic_pd:.4f}')
print(f'XGBoost per-date mean IC (ex Fold 1 COVID)          : {mean_ic_ex_covid:.4f}')
print(f'XGBoost per-date mean IC (ex Fold 1 + Fold 6)       : {mean_ic_ex_covid_f6:.4f}')
print(f'XGBoost per-date mean IC (ex Folds 1,6,8 — honest)  : {mean_ic_honest:.4f}  ← primary baseline')
print(f'  Normal market folds used: {[r["fold"] for r in xgb_cv_results if r["fold"] not in REGIME_BREAK_FOLDS]}')

if   mean_ic_honest > 0.06: grade = '🟢 STRONG  — IC > 0.06, production-grade signal'
elif mean_ic_honest > 0.04: grade = '🟡 GOOD    — IC 0.04–0.06, usable signal'
elif mean_ic_honest > 0.02: grade = '🟠 WEAK    — IC 0.02–0.04, marginal, review features'
else:                       grade = '🔴 POOR    — IC < 0.02, investigate data/features'

print(f'\nSignal grade (honest baseline): {grade}')
print(f'\nNOTE: Regime-break folds excluded from baseline:')
print(f'  Fold 1 = COVID crash/recovery (2020) — anomalous momentum, not representative')
print(f'  Fold 6 = 2022 RBI/Fed rate-shock — momentum reversal regime')
print(f'  Fold 8 = 2023 sideways chop — mean-reversion dominated, momentum anti-predictive')

WALK-FORWARD CV RESULTS
  Fold   XGB Global  LGB Global  XGB PerDate  LGB PerDate  XGB ICIR  LGB ICIR
  -----------------------------------------------------------------------
  1         +0.1323     +0.1214      +0.1397      +0.1268     0.939     0.846 ← COVID (regime-break, excluded from honest baseline)
  2         +0.0175     +0.0172      +0.0179      +0.0187     0.272     0.298
  3         +0.0503     +0.0464      +0.0457      +0.0428     0.564     0.557
  4         +0.0269     +0.0385      +0.0233      +0.0357     0.202     0.320
  5         +0.0299     +0.0252      +0.0279      +0.0235     0.342     0.268
  6         -0.0773     -0.0616      -0.0765      -0.0620    -0.763    -0.617 ← 2022 rate-shock (regime-break, excluded from honest baseline)
  7         +0.0206     +0.0174      +0.0188      +0.0163     0.251     0.221
  8         -0.0089     -0.0059      -0.0091      -0.0062    -0.097    -0.073 ← 2023 chop (regime-break, excluded from honest baseline)
  ----------------------

## Cell 14 — IC per Fold Plots

In [17]:
fold_nums = [r['fold'] for r in xgb_cv_results]
x     = np.arange(len(fold_nums))
width = 0.35
BLUE  = '#2980b9'
GREEN = '#27ae60'

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Walk-Forward CV — XGBoost vs LightGBM (v5)', fontsize=13, y=1.01)

for ax, (xgb_vals, lgb_vals), title, ylabel in [
    (axes[0], (xgb_ics, lgb_ics),
     'Global Spearman IC per Fold', 'IC (Spearman)'),
    (axes[1], (xgb_mean_ics, lgb_mean_ics),
     'Per-date Mean IC per Fold', 'Per-date mean IC'),
]:
    b1 = ax.bar(x - width/2, xgb_vals, width, label='XGBoost',
                color=BLUE,  edgecolor='black', linewidth=0.5)
    b2 = ax.bar(x + width/2, lgb_vals, width, label='LightGBM',
                color=GREEN, edgecolor='black', linewidth=0.5)
    ax.axhline(np.nanmean(xgb_vals), color=BLUE,  linestyle='--',
               linewidth=1.2, alpha=0.8,
               label=f'XGB μ={np.nanmean(xgb_vals):.4f}')
    ax.axhline(np.nanmean(lgb_vals), color=GREEN, linestyle='--',
               linewidth=1.2, alpha=0.8,
               label=f'LGB μ={np.nanmean(lgb_vals):.4f}')
    ax.axhline(0, color='black', linewidth=0.8)
    for bar in b1:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + (0.001 if bar.get_height() >= 0 else -0.003),
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    ax.set_xlabel('Fold')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels([f'F{n}' for n in fold_nums])
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
p = os.path.join(RESULTS_DIR, f'ic_per_fold_{DATE_STAMP}.png')
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

Saved: /content/drive/MyDrive/Hedge_Fund_Simulator_V2/results/xgboost_v5/ic_per_fold_20260502.png


## Cell 15 — Feature Importance (Gain-Based)

In [18]:
TOP_N = 25

xgb_last  = xgb_cv_results[-1]['model']
lgb_last  = lgb_cv_results[-1]['model']
last_fold = xgb_cv_results[-1]['fold']
print(f'Using fold {last_fold} models for importance plots')

xgb_gain_raw = xgb_last.get_booster().get_score(importance_type='gain')
xgb_imp      = (pd.Series(xgb_gain_raw)
                  .reindex(FEATURE_COLS)
                  .fillna(0)
                  .sort_values(ascending=False)
                  .head(TOP_N))

fig, ax = plt.subplots(figsize=(10, 8))
xgb_imp[::-1].plot(kind='barh', ax=ax, color=BLUE, edgecolor='black', linewidth=0.5)
ax.set_title(f'XGBoost — Top {TOP_N} Features by Gain (Fold {last_fold}, v5)')
ax.set_xlabel('Gain (total loss reduction across all splits)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
p = os.path.join(RESULTS_DIR, f'xgb_importance_gain_{DATE_STAMP}.png')
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

lgb_imp = (pd.Series(lgb_last.feature_importances_, index=FEATURE_COLS)
             .sort_values(ascending=False)
             .head(TOP_N))

fig, ax = plt.subplots(figsize=(10, 8))
lgb_imp[::-1].plot(kind='barh', ax=ax, color=GREEN, edgecolor='black', linewidth=0.5)
ax.set_title(f'LightGBM — Top {TOP_N} Features by Gain (Fold {last_fold}, v5)')
ax.set_xlabel('Gain')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
p = os.path.join(RESULTS_DIR, f'lgb_importance_gain_{DATE_STAMP}.png')
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

print(f'\n{"Top 10 XGBoost features (gain)":<45} {"Top 10 LightGBM features (gain)"}')
print(f'{"-"*44} {"-"*44}')
for (xf, xv), (lf, lv) in zip(xgb_imp.head(10).items(), lgb_imp.head(10).items()):
    print(f'  {xf:<38} {xv:>8.1f}   {lf:<38} {lv:>8.1f}')

for flag in FLAG_COLS:
    assert flag not in xgb_gain_raw, f'FLAG COLUMN {flag} IN XGB IMPORTANCE — BUG'
print('\n✅ No flag columns in feature importance.')

Using fold 8 models for importance plots
Saved: /content/drive/MyDrive/Hedge_Fund_Simulator_V2/results/xgboost_v5/xgb_importance_gain_20260502.png
Saved: /content/drive/MyDrive/Hedge_Fund_Simulator_V2/results/xgboost_v5/lgb_importance_gain_20260502.png

Top 10 XGBoost features (gain)                Top 10 LightGBM features (gain)
-------------------------------------------- --------------------------------------------
  Gross_Profit_Margin                         3.7   ATR_14                                   7566.7
  Revenue_YoY_Growth                          3.4   Revenue_YoY_Growth                       6675.7
  EBITDA_Margin                               3.3   Beta_to_Crude                            6488.5
  Debt_to_Assets                              3.3   Revenue_Growth_Sector_Rank               6399.6
  Rel_EBITDA_Margin                           3.2   Beta_to_USDINR                           6072.2
  Rel_ROA                                     3.2   Beta_to_VIX               

## Cell 16 — Retrain Final Models on Full Training Pool

In [19]:
print('Retraining final models on full training pool...')
print(f'  Pool: {df[DATE_COL].min().date()} → {df[DATE_COL].max().date()}'
      f' | {len(df):,} rows')

FINAL_MEDIANS = compute_impute_medians(df)
X_full_raw    = apply_imputation(df[FEATURE_COLS], df, FINAL_MEDIANS)
y_full        = df[TARGET_COL].values

active_scale  = [c for c in SCALE_COLS if c in X_full_raw.columns]
X_full_winsr  = winsorize_dataframe(X_full_raw, active_scale, WINSOR_LIMITS)

scaler_final  = StandardScaler()
X_full        = X_full_winsr.copy()
X_full[active_scale] = scaler_final.fit_transform(X_full_winsr[active_scale])

null_after = X_full.isna().sum().sum()
print(f'  Residual NULLs after imputation: {null_after:,}  '
      f'(XGBoost handles these natively via its missing= parameter)')

final_weights = compute_time_decay_weights(df[DATE_COL], DECAY_HALF_LIFE_FINAL)
print(f'  Sample weight range: [{final_weights.min():.3f}, {final_weights.max():.3f}]'
      f'  mean={final_weights.mean():.3f}')

xgb_params_final = XGB_PARAMS.copy()
try:
    xgb_final = xgb.XGBRegressor(**xgb_params_final)
    xgb_final.fit(X_full, y_full, sample_weight=final_weights, verbose=False)
    print('  XGBoost final: GPU')
except Exception as e:
    print(f'  XGBoost GPU failed ({e}), using CPU...')
    xgb_params_final['device'] = 'cpu'
    xgb_final = xgb.XGBRegressor(**xgb_params_final)
    xgb_final.fit(X_full, y_full, sample_weight=final_weights, verbose=False)

lgb_params_final = LGB_PARAMS.copy()
try:
    lgb_final = lgb.LGBMRegressor(**lgb_params_final)
    lgb_final.fit(X_full, y_full, sample_weight=final_weights)
    print('  LightGBM final: GPU')
except Exception as e:
    print(f'  LightGBM GPU failed ({e}), using CPU...')
    lgb_params_final['device'] = 'cpu'
    lgb_final = lgb.LGBMRegressor(**lgb_params_final)
    lgb_final.fit(X_full, y_full, sample_weight=final_weights)

print('\n✅ Final models trained on full pool with time-decay weights.')

Retraining final models on full training pool...
  Pool: 2010-01-04 → 2023-12-29 | 1,023,895 rows
  Residual NULLs after imputation: 4,092,097  (XGBoost handles these natively via its missing= parameter)
  Sample weight range: [0.037, 4.001]  mean=1.000
  XGBoost final: GPU
  LightGBM final: GPU

✅ Final models trained on full pool with time-decay weights.


## Cell 17 — Holdout Evaluation (Jan 2024 → Present)

In [20]:
print('=' * 60)
print('HOLDOUT EVALUATION — Jan 2024 → present')
print('=' * 60)

h_df = holdout_df[holdout_df[TARGET_COL].notna()].copy()
print(f'Holdout rows   : {len(h_df):,}')
print(f'Holdout dates  : {h_df[DATE_COL].min().date()} → {h_df[DATE_COL].max().date()}')
print(f'Holdout tickers: {h_df[TICKER_COL].nunique()}')

for col in FEATURE_COLS:
    if col in h_df.columns and h_df[col].dtype == object:
        h_df[col] = pd.to_numeric(h_df[col], errors='coerce')

X_hold_raw  = apply_imputation(h_df[FEATURE_COLS], h_df, FINAL_MEDIANS)
X_hold_winsr = winsorize_dataframe(X_hold_raw, active_scale, WINSOR_LIMITS)
X_hold      = X_hold_winsr.copy()
X_hold[active_scale] = scaler_final.transform(X_hold_winsr[active_scale])
y_hold      = h_df[TARGET_COL].values

xgb_h_preds = xgb_final.predict(X_hold)
lgb_h_preds = lgb_final.predict(X_hold)

xgb_h_ic_global = compute_ic(y_hold, xgb_h_preds)
lgb_h_ic_global = compute_ic(y_hold, lgb_h_preds)

_hx = h_df[[DATE_COL, TARGET_COL]].copy(); _hx['pred'] = xgb_h_preds
_hl = h_df[[DATE_COL, TARGET_COL]].copy(); _hl['pred'] = lgb_h_preds
xgb_h_mean_ic, xgb_h_icir, xgb_h_daily = compute_ic_per_date(_hx)
lgb_h_mean_ic, lgb_h_icir, lgb_h_daily = compute_ic_per_date(_hl)

print(f'\n  {"Model":<12} {"Global IC":>10} {"Per-date IC":>12} {"ICIR":>8}')
print(f'  {"-"*44}')
print(f'  {"XGBoost":<12} {xgb_h_ic_global:>+10.4f} {xgb_h_mean_ic:>+12.4f} {xgb_h_icir:>8.3f}')
print(f'  {"LightGBM":<12} {lgb_h_ic_global:>+10.4f} {lgb_h_mean_ic:>+12.4f} {lgb_h_icir:>8.3f}')
print('=' * 60)

if mean_ic_honest != 0 and not np.isnan(mean_ic_honest):
    ic_decay = (mean_ic_honest - xgb_h_mean_ic) / abs(mean_ic_honest)
else:
    ic_decay = np.nan

if np.isnan(ic_decay):
    decay_flag = '⚠️  Cannot compute (honest baseline is NaN or zero)'
elif ic_decay > 0.30:
    decay_flag = '⚠️  HIGH DECAY — holdout significantly worse than normal-market CV'
elif ic_decay < 0:
    decay_flag = '✅ IMPROVEMENT — holdout better than normal-market CV baseline'
else:
    decay_flag = '✅ ACCEPTABLE — decay within 30%'

print(f'\nIC decay (honest CV baseline → holdout): {ic_decay:+.1%}  {decay_flag}')
print(f'  Interpretation: positive = degradation, negative = improvement')
print(f'  Honest baseline (ex Folds 1,6,8): {mean_ic_honest:.4f}  |  holdout: {xgb_h_mean_ic:.4f}')

fig, ax = plt.subplots(figsize=(14, 4))
xgb_h_daily.plot(ax=ax, color=BLUE,  alpha=0.7, label=f'XGBoost (μ={xgb_h_mean_ic:.4f})')
lgb_h_daily.plot(ax=ax, color=GREEN, alpha=0.7, label=f'LightGBM (μ={lgb_h_mean_ic:.4f})')
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(xgb_h_mean_ic, color=BLUE,  linestyle='--', linewidth=1)
ax.axhline(lgb_h_mean_ic, color=GREEN, linestyle='--', linewidth=1)
ax.set_xlabel('Date')
ax.set_ylabel('Daily IC')
ax.set_title('Holdout Daily IC (Jan 2024 → present) — v5')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
p = os.path.join(RESULTS_DIR, f'holdout_daily_ic_{DATE_STAMP}.png')
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

HOLDOUT EVALUATION — Jan 2024 → present
Holdout rows   : 206,472
Holdout dates  : 2024-01-01 → 2026-03-23
Holdout tickers: 382

  Model         Global IC  Per-date IC     ICIR
  --------------------------------------------
  XGBoost         +0.0338      +0.0312    0.276
  LightGBM        +0.0411      +0.0388    0.354

IC decay (honest CV baseline → holdout): -16.6%  ✅ IMPROVEMENT — holdout better than normal-market CV baseline
  Interpretation: positive = degradation, negative = improvement
  Honest baseline (ex Folds 1,6,8): 0.0267  |  holdout: 0.0312
Saved: /content/drive/MyDrive/Hedge_Fund_Simulator_V2/results/xgboost_v5/holdout_daily_ic_20260502.png


## Cell 18 — Save Models to Drive

In [21]:
common_meta = {
    'feature_cols'          : FEATURE_COLS,
    'scale_cols'            : SCALE_COLS,
    'target_col'            : TARGET_COL,
    'cv_ics'                : None,
    'cv_mean_ics'           : None,
    'cv_icirs'              : None,
    'holdout_ic'            : None,
    'holdout_mean_ic'       : None,
    'holdout_icir'          : None,
    'date_stamp'            : DATE_STAMP,
    'holdout_start'         : str(HOLDOUT_START.date()),
    'train_years'           : TRAIN_YEARS,
    'n_folds'               : N_FOLDS,
    'purge_days'            : PURGE_DAYS,
    'decay_half_life_final' : DECAY_HALF_LIFE_FINAL,
    'decay_half_life_fold'  : DECAY_HALF_LIFE_FOLD,
    'winsor_limits'         : WINSOR_LIMITS,
    'excluded_cols'         : sorted(list(EXCLUDE_COLS)),
    'flag_impute_map'       : {k: v[1] for k, v in FLAG_IMPUTE_MAP.items()},
    'impute_medians'        : {str(k): v for k, v in FINAL_MEDIANS.items()},
    'scaler'                : scaler_final,
    'cv_mean_ic_ex_covid'   : float(np.nanmean(xgb_mean_ics[1:])),
    'ic_decay'              : float(ic_decay) if not np.isnan(ic_decay) else None,
    'version'               : 'v5',
}

xgb_payload = {**common_meta,
    'model'          : xgb_final,
    'params'         : XGB_PARAMS,
    'cv_ics'         : xgb_ics,
    'cv_mean_ics'    : xgb_mean_ics,
    'cv_icirs'       : xgb_icirs,
    'holdout_ic'     : float(xgb_h_ic_global),
    'holdout_mean_ic': float(xgb_h_mean_ic),
    'holdout_icir'   : float(xgb_h_icir),
}
xgb_path = os.path.join(MODELS_DIR, f'xgboost_v5_{DATE_STAMP}.pkl')
with open(xgb_path, 'wb') as f:
    pickle.dump(xgb_payload, f)
print(f'Saved: {xgb_path}  ({os.path.getsize(xgb_path)/1e6:.1f} MB)')

lgb_payload = {**common_meta,
    'model'          : lgb_final,
    'params'         : LGB_PARAMS,
    'cv_ics'         : lgb_ics,
    'cv_mean_ics'    : lgb_mean_ics,
    'cv_icirs'       : lgb_icirs,
    'holdout_ic'     : float(lgb_h_ic_global),
    'holdout_mean_ic': float(lgb_h_mean_ic),
    'holdout_icir'   : float(lgb_h_icir),
}
lgb_path = os.path.join(MODELS_DIR, f'lightgbm_v5_{DATE_STAMP}.pkl')
with open(lgb_path, 'wb') as f:
    pickle.dump(lgb_payload, f)
print(f'Saved: {lgb_path}  ({os.path.getsize(lgb_path)/1e6:.1f} MB)')

print('\n✅ Both models saved.')

Saved: /content/drive/MyDrive/Hedge_Fund_Simulator_V2/models/xgboost_v5_20260502.pkl  (4.9 MB)
Saved: /content/drive/MyDrive/Hedge_Fund_Simulator_V2/models/lightgbm_v5_20260502.pkl  (6.9 MB)

✅ Both models saved.


## Cell 19 — Sanity Checks

In [22]:
print('=' * 65)
print('SANITY CHECKS')
print('=' * 65)
results = []

def chk(label, passed, detail=''):
    flag = '✅ PASS' if passed else '❌ FAIL'
    results.append(passed)
    print(f'  [{flag}] {label}')
    if detail:
        print(f'          {detail}')

chk('Both models positive mean IC',
    np.mean(xgb_ics) > 0 and np.mean(lgb_ics) > 0,
    f'XGB={np.mean(xgb_ics):.4f}  LGB={np.mean(lgb_ics):.4f}')

non_regime_break_ics = [
    r['ic'] for r in xgb_cv_results
    if r['fold'] not in REGIME_BREAK_FOLDS
]
min_normal_xgb = min(non_regime_break_ics) if non_regime_break_ics else 0
regime_break_folds_negative = [
    r['fold'] for r in xgb_cv_results
    if r['fold'] in REGIME_BREAK_FOLDS and r['ic'] < -0.02
]
chk('No normal fold IC < -0.02 (regime-break folds 1,6,8 expected negative)',
    min_normal_xgb > -0.02,
    f'Normal fold min={min_normal_xgb:.4f} | '
    f'Regime-break folds < -0.02: {regime_break_folds_negative} (expected)')

xgb_nf = len(xgb_final.feature_importances_)
chk(f'XGBoost feature count matches ({xgb_nf} == {len(FEATURE_COLS)})',
    xgb_nf == len(FEATURE_COLS))

leaked = [c for c in FLAG_COLS if c in FEATURE_COLS]
chk('No flag columns in FEATURE_COLS',
    len(leaked) == 0,
    f'Leaked: {leaked}' if leaked else '')

leaked_p = [c for c in RAW_PRICE_COLS if c in FEATURE_COLS]
chk('No raw price/volume levels in FEATURE_COLS',
    len(leaked_p) == 0,
    f'Leaked: {leaked_p}' if leaked_p else '')

chk('Holdout dates strictly after training pool',
    df[DATE_COL].max() < HOLDOUT_START)

purge_ok = all(f['train_end'] < f['test_start'] for f in folds)
chk(f'21-day purge gap enforced in all {len(folds)} folds', purge_ok)

for path in [xgb_path, lgb_path]:
    exists = os.path.exists(path) and os.path.getsize(path) > 1000
    chk(f'Model file exists: {os.path.basename(path)}', exists)

PRED_MIN_THRESH = -0.15
PRED_MAX_THRESH =  1.15
chk(f'XGBoost holdout prediction range in [{PRED_MIN_THRESH}, {PRED_MAX_THRESH}]',
    xgb_h_preds.min() > PRED_MIN_THRESH and xgb_h_preds.max() < PRED_MAX_THRESH,
    f'Range: [{xgb_h_preds.min():.4f}, {xgb_h_preds.max():.4f}]  '
    f'(minor extrapolation outside [0,1] is normal for regression on rank targets)')

if not np.isnan(ic_decay):
    chk(f'Holdout IC decay < 30% vs honest baseline (actual: {ic_decay:+.1%}) [FIX-C1]',
        ic_decay < 0.30,
        f'Honest CV baseline (ex Folds 1,6,8): {mean_ic_honest:.4f} | '
        f'holdout: {xgb_h_mean_ic:.4f} | negative = improvement')
else:
    chk('Holdout IC decay check (SKIPPED — honest baseline unavailable)', True)

chk('No NaN in XGBoost holdout predictions',
    not np.isnan(xgb_h_preds).any())

chk(f'Final sample weights mean ≈ 1.0 (actual: {final_weights.mean():.4f})',
    abs(final_weights.mean() - 1.0) < 0.01)

xgb_gain_raw_final = xgb_final.get_booster().get_score(importance_type='gain')
flag_leak = [f for f in FLAG_COLS if f in xgb_gain_raw_final]
chk('No flag columns in XGBoost gain importance',
    len(flag_leak) == 0,
    f'Leaked flags: {flag_leak}' if flag_leak else '')

new_present = [c for c in NEW_V23_COLS if c in FEATURE_COLS]
chk(f'New v2.3 feature columns present ({len(new_present)}/9)',
    len(new_present) == 9,
    f'Missing: {[c for c in NEW_V23_COLS if c not in FEATURE_COLS]}' if len(new_present) < 9 else '')

rank_in_scale = [c for c in SCALE_COLS if 'Rank' in c or c in ('Mom_12_1', 'Mom_6_1')]
chk('Rank/momentum columns not in SCALE_COLS (FIX-B)',
    len(rank_in_scale) == 0,
    f'In scale: {rank_in_scale}' if rank_in_scale else '')

print()
passed = sum(results)
total  = len(results)
status = '✅ ALL PASSED' if passed == total else f'⚠️  {total-passed} FAILED'
print(f'  Result: {passed}/{total} checks passed  —  {status}')
print('=' * 65)
print(f'\nXGBoost  : xgboost_v5_{DATE_STAMP}.pkl')
print(f'LightGBM : lightgbm_v5_{DATE_STAMP}.pkl')
print(f'Saved to : {MODELS_DIR}')
print('\nNext step: hedge_fund_lstm_v2.ipynb')

SANITY CHECKS
  [✅ PASS] Both models positive mean IC
          XGB=0.0239  LGB=0.0248
  [✅ PASS] No normal fold IC < -0.02 (regime-break folds 1,6,8 expected negative)
          Normal fold min=0.0175 | Regime-break folds < -0.02: [6] (expected)
  [✅ PASS] XGBoost feature count matches (60 == 60)
  [✅ PASS] No flag columns in FEATURE_COLS
  [✅ PASS] No raw price/volume levels in FEATURE_COLS
  [✅ PASS] Holdout dates strictly after training pool
  [✅ PASS] 21-day purge gap enforced in all 8 folds
  [✅ PASS] Model file exists: xgboost_v5_20260502.pkl
  [✅ PASS] Model file exists: lightgbm_v5_20260502.pkl
  [✅ PASS] XGBoost holdout prediction range in [-0.15, 1.15]
          Range: [-0.0683, 0.9989]  (minor extrapolation outside [0,1] is normal for regression on rank targets)
  [✅ PASS] Holdout IC decay < 30% vs honest baseline (actual: -16.6%) [FIX-C1]
          Honest CV baseline (ex Folds 1,6,8): 0.0267 | holdout: 0.0312 | negative = improvement
  [✅ PASS] No NaN in XGBoost holdout pr